# Code

In [1]:
# ============================================================
# Improved MABe Social Behavior Detection with XGBoost
# Improved inference notebook (fold aggregation + postprocessing)
# ============================================================

from pathlib import Path
import os
import sys

# ------------------------------------------------------------
# Input dataset checks
# ------------------------------------------------------------
COMP_DIR = Path("/kaggle/input/MABe-mouse-behavior-detection")
STARTER_DIR = Path("/kaggle/input/mabe-starter-train-ja")
MABE_PKG_DIR = Path("/kaggle/input/mabe-package")

if not COMP_DIR.exists():
    raise FileNotFoundError(
        "Competition dataset 'MABe Challenge - Social Action Recognition in Mice' "
        "must be attached as an input."
    )

if not STARTER_DIR.exists():
    raise FileNotFoundError(
        "Dataset 'mabe-starter-train-ja' is not attached. "
        "Click 'Add input' and add it before running."
    )

if not MABE_PKG_DIR.exists():
    raise FileNotFoundError(
        "Dataset 'mabe-package' is not attached. "
        "It provides the offline xgboost wheel used by the starter models."
    )

# ------------------------------------------------------------
# Install xgboost from offline wheel (no internet)
# ------------------------------------------------------------
!pip install -q --no-index --find-links=/kaggle/input/mabe-package xgboost==3.1.1

# ------------------------------------------------------------
# Copy helper scripts and trained models from starter dataset
# ------------------------------------------------------------
!cp /kaggle/input/mabe-starter-train-ja/self_features.py .
!cp /kaggle/input/mabe-starter-train-ja/pair_features.py .
!cp -r /kaggle/input/mabe-starter-train-ja/results .

# ============================================================
# Imports
# ============================================================
import gc
import re
import ast
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import json

# polars is preinstalled on Kaggle GPU/CPU images
try:
    import polars as pl
except ImportError:
    raise ImportError(
        "polars is not available in this environment. "
        "Use a Kaggle GPU/CPU notebook image where polars is preinstalled."
    )

import xgboost as xgb
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import multiprocessing

# Helper scripts from starter notebook
%run -i self_features.py
%run -i pair_features.py

# Global setting to adjust sensitivity (Lower = More Recall)
THRESHOLD_FACTOR = 1.0

# CPU-friendly parallelism helper
def get_n_jobs():
    try:
        env_val = os.getenv("N_JOBS")
        if env_val:
            return max(1, int(env_val))
    except Exception:
        pass
    # Kaggle CPU kernels often have limited resources; be conservative without GPU
    if os.getenv("KAGGLE_KERNEL_RUN_TYPE") is not None and not os.path.exists("/usr/bin/nvidia-smi"):
        return 2
    return max(1, multiprocessing.cpu_count() - 1)


def robustify(submission: pl.DataFrame, dataset: pl.DataFrame, train_test: str = "train"):
    traintest_directory = INPUT_DIR / f"{train_test}_tracking"

    old_submission = submission.clone()
    submission = submission.filter(pl.col("start_frame") < pl.col("stop_frame"))
    if len(submission) != len(old_submission):
        print("ERROR: Dropped frames with start >= stop")

    old_submission = submission.clone()
    group_list = []
    for _, group in submission.group_by("video_id", "agent_id", "target_id"):
        group = group.sort("start_frame")
        mask = np.ones(len(group), dtype=bool)
        last_stop_frame = 0
        for i, row in enumerate(group.rows(named=True)):
            if row["start_frame"] < last_stop_frame:
                mask[i] = False
            else:
                last_stop_frame = row["stop_frame"]
        group_list.append(group.filter(pl.Series("mask", mask)))

    submission = pl.concat(group_list)

    if len(submission) != len(old_submission):
        print("ERROR: Dropped duplicate frames")

    s_list = []
    for row in dataset.rows(named=True):
        lab_id = row["lab_id"]
        video_id = row["video_id"]
        if row["behaviors_labeled"] is None:
            continue

        if video_id in submission.get_column("video_id").to_list():
            continue

        if isinstance(row["behaviors_labeled"], str):
            continue

        print(f"Video {video_id} has no predictions.")

        path = traintest_directory / f"/{lab_id}/{video_id}.parquet"
        vid = pd.read_parquet(path)

        vid_behaviors = json.loads(row["behaviors_labeled"])
        vid_behaviors = sorted(list({b.replace("'", "") for b in vid_behaviors}))
        vid_behaviors = [b.split(",") for b in vid_behaviors]
        vid_behaviors = pd.DataFrame(vid_behaviors, columns=["agent", "target", "action"])

        start_frame = vid.video_frame.min()
        stop_frame = vid.video_frame.max() + 1

        for (agent, target), actions in vid_behaviors.groupby(["agent", "target"]):
            batch_length = int(np.ceil((stop_frame - start_frame) / len(actions)))
            for i, action_row in enumerate(actions.itertuples(index=False)):
                batch_start = start_frame + i * batch_length
                batch_stop = min(batch_start + batch_length, stop_frame)
                s_list.append((video_id, agent, target, action_row["action"], batch_start, batch_stop))

    if len(s_list) > 0:
        submission = pl.concat(
            [
                submission,
                pl.DataFrame(s_list, schema=["video_id", "agent_id", "target_id", "action", "start_frame", "stop_frame"], orient="row"),
            ],
            how="vertical"
        )
        print("ERROR: Filled empty videos")

    return submission

# ============================================================
# Paths and constants
# ============================================================
INPUT_DIR = COMP_DIR
TRAIN_TRACKING_DIR = INPUT_DIR / "train_tracking"
TRAIN_ANNOTATION_DIR = INPUT_DIR / "train_annotation"
TEST_TRACKING_DIR = INPUT_DIR / "test_tracking"

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

SELF_FEATURE_DIR = WORKING_DIR / "self_features"
PAIR_FEATURE_DIR = WORKING_DIR / "pair_features"
SELF_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
PAIR_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

INDEX_COLS = [
    "video_id",
    "agent_mouse_id",
    "target_mouse_id",
    "video_frame",
]

BODY_PARTS = [
    "ear_left",
    "ear_right",
    "nose",
    "neck",
    "body_center",
    "lateral_left",
    "lateral_right",
    "hip_left",
    "hip_right",
    "tail_base",
    "tail_tip",
]

SELF_BEHAVIORS = [
    "biteobject",
    "climb",
    "dig",
    "exploreobject",
    "freeze",
    "genitalgroom",
    "huddle",
    "rear",
    "rest",
    "run",
    "selfgroom",
]

PAIR_BEHAVIORS = [
    "allogroom",
    "approach",
    "attack",
    "attemptmount",
    "avoid",
    "chase",
    "chaseattack",
    "defend",
    "disengage",
    "dominance",
    "dominancegroom",
    "dominancemount",
    "ejaculate",
    "escape",
    "flinch",
    "follow",
    "intromit",
    "mount",
    "reciprocalsniff",
    "shepherd",
    "sniff",
    "sniffbody",
    "sniffface",
    "sniffgenital",
    "submit",
    "tussle",
]

# ============================================================
# Helper functions
# ============================================================

def parse_behaviors_column(behaviors_str: str):
    """
    behaviors_labeled is stored as a Python like list of tuples.
    Use ast.literal_eval for safety instead of eval.

    Example:
      "[('mouse1','mouse2','sniff'), ('mouse2','mouse1','sniff')]"
    """
    if behaviors_str is None:
        return []
    return ast.literal_eval(behaviors_str)


def build_behavior_dataframe(test_df: pl.DataFrame) -> pl.DataFrame:
    """
    Expand behaviors_labeled into one row per (lab, video, agent, target, behavior).
    """
    behavior_df = (
        test_df
        .filter(pl.col("behaviors_labeled").is_not_null())
        .select(["lab_id", "video_id", "behaviors_labeled"])
        .with_columns(
            pl.col("behaviors_labeled")
            .map_elements(
                parse_behaviors_column,
                return_dtype=pl.List(pl.Utf8),
            )
            .alias("behaviors_labeled_list")
        )
        .explode("behaviors_labeled_list")
        .rename({"behaviors_labeled_list": "behaviors_labeled_element"})
        .with_columns(
            pl.col("behaviors_labeled_element").str.split(",").list.get(0)
            .str.replace_all("[()' ]", "")
            .alias("agent"),
            pl.col("behaviors_labeled_element").str.split(",").list.get(1)
            .str.replace_all("[()' ]", "")
            .alias("target"),
            pl.col("behaviors_labeled_element").str.split(",").list.get(2)
            .str.replace_all("[()' ]", "")
            .alias("behavior"),
        )
        .select(["lab_id", "video_id", "agent", "target", "behavior"])
    )
    return behavior_df


def extract_mouse_id(mouse_str: str) -> int:
    """
    Convert 'mouse1' -> 1, 'mouse2' -> 2, 'self' -> -1.
    """
    if mouse_str == "self":
        return -1
    m = re.search(r"mouse(\d+)", mouse_str)
    if m:
        return int(m.group(1))
    raise ValueError(f"Unexpected mouse id format: {mouse_str}")


def load_features_for_group(lab_id, video_id, agent, target):
    """
    Load per frame features for a given (lab, video, agent, target) group.
    Returns:
      index_df   - DataFrame with INDEX_COLS
      feature_df - DataFrame with feature columns only
    """
    agent_mouse_id = extract_mouse_id(agent)
    target_mouse_id = extract_mouse_id(target)

    if target == "self":
        feature_path = SELF_FEATURE_DIR / f"{video_id}.parquet"
        scan = pl.scan_parquet(feature_path).filter(
            pl.col("agent_mouse_id") == agent_mouse_id
        )
    else:
        feature_path = PAIR_FEATURE_DIR / f"{video_id}.parquet"
        scan = pl.scan_parquet(feature_path).filter(
            (pl.col("agent_mouse_id") == agent_mouse_id)
            & (pl.col("target_mouse_id") == target_mouse_id)
        )

    full_df = scan.collect()
    if full_df.height == 0:
        return full_df, full_df

    index_df = full_df.select(INDEX_COLS)
    feature_df = full_df.select(pl.exclude(INDEX_COLS))
    return index_df, feature_df


def load_models_for_behavior(lab_id: str, behavior: str):
    """
    Load all fold models and thresholds for a given (lab, behavior).
    Returns list of (model, threshold).
    """
    behavior_dir = WORKING_DIR / "results" / lab_id / behavior
    fold_dirs = sorted(behavior_dir.glob("fold_*"))
    models = []
    for fold_dir in fold_dirs:
        model_file = fold_dir / "model.json"
        thr_file = fold_dir / "threshold.txt"
        if not model_file.exists() or not thr_file.exists():
            continue
        with open(thr_file, "r") as f:
            threshold = float(f.read().strip()) * THRESHOLD_FACTOR
        model = xgb.Booster(model_file=str(model_file))
        models.append((model, threshold))
    return models


def predict_for_group(
    lab_id: str,
    video_id: int,
    agent: str,
    target: str,
    group_behaviors: pl.DataFrame,
):
    """
    Run inference for one group of (lab_id, video_id, agent, target).

    Improvements:
      - Aggregate folds per behavior by averaging probabilities.
      - Apply a single per-behavior threshold (mean of fold thresholds).
      - Pick best behavior per frame among candidates above threshold; otherwise 'none'.
    """
    index_df, feature_df = load_features_for_group(lab_id, video_id, agent, target)

    if feature_df.height == 0:
        return None

    # Create XGBoost DMatrix once per group and reuse across behaviors
    dtest = xgb.DMatrix(feature_df.to_pandas(), feature_names=feature_df.columns)

    prediction_df = index_df.clone()
    used_cols = []

    # Unique behaviors for this group
    unique_behaviors = (
        group_behaviors.select("behavior").unique()["behavior"].to_list()
    )

    beh_threshold = {}

    for behavior in unique_behaviors:
        models = load_models_for_behavior(lab_id, behavior)
        if not models:
            continue

        # Average probabilities across folds; single threshold per behavior
        probs_sum = np.zeros(feature_df.height, dtype=np.float32)
        thr_vals = []
        for model, threshold in models:
            probs = model.predict(dtest)
            probs_sum += probs
            thr_vals.append(threshold)
        avg_probs = probs_sum / max(len(models), 1)
        avg_thr = float(np.mean(thr_vals)) if thr_vals else 0.5

        col_name = behavior
        prediction_df = prediction_df.with_columns(
            pl.Series(name=col_name, values=avg_probs)
        )
        used_cols.append(col_name)
        beh_threshold[col_name] = avg_thr

    if not used_cols:
        return None

    # Pick best behavior per frame among those above per-behavior threshold
    cols = used_cols
    thr_vec = np.array([beh_threshold[c] for c in cols], dtype=np.float32)

    # Build labels by selecting highest score if any behavior passes its threshold
    def choose_label(row):
        vals = np.array(list(row.values()), dtype=np.float32)
        mask = vals >= thr_vec
        if mask.any():
            return cols[int(np.argmax(vals))]
        return "none"

    prediction_labels_df = (
        prediction_df
        .with_columns(
            pl.struct(pl.col(cols))
            .map_elements(
                choose_label,
                return_dtype=pl.String,
            )
            .alias("prediction")
        )
        .select(INDEX_COLS + ["prediction"])
    )

    # --- Gap Filling: Fill 1-frame gaps ---
    prediction_labels_df = prediction_labels_df.with_columns(
        pl.when(
            (pl.col("prediction") == "none") &
            (pl.col("prediction").shift(1) == pl.col("prediction").shift(-1)) &
            (pl.col("prediction").shift(1).is_not_null()) &
            (pl.col("prediction").shift(1) != "none")
        )
        .then(pl.col("prediction").shift(1))
        .otherwise(pl.col("prediction"))
        .alias("prediction")
    )

    # Convert per frame labels into time segments
    agent_mouse_id = extract_mouse_id(agent)
    target_mouse_id = extract_mouse_id(target)

    group_submission = (
        prediction_labels_df
        .filter(pl.col("prediction") != pl.col("prediction").shift(1))
        .with_columns(
            pl.col("video_frame").shift(-1).alias("stop_frame")
        )
        .filter(pl.col("prediction") != "none")
        .select(
            pl.col("video_id"),
            (pl.lit("mouse") + pl.lit(agent_mouse_id).cast(pl.Utf8)).alias("agent_id"),
            pl.when(pl.lit(target_mouse_id) == -1)
            .then(pl.lit("self"))
            .otherwise(pl.lit("mouse") + pl.lit(target_mouse_id).cast(pl.Utf8))
            .alias("target_id"),
            pl.col("prediction").alias("action"),
            pl.col("video_frame").alias("start_frame"),
            pl.col("stop_frame"),
        )
    )

    return group_submission

# ============================================================
# 1. Load metadata and build behavior table
# ============================================================
print("Loading test metadata...")
test_df = pl.read_csv(INPUT_DIR / "test.csv")

print("Building behavior table from behaviors_labeled...")
behavior_df = build_behavior_dataframe(test_df)

groups = list(
    behavior_df.group_by("lab_id", "video_id", "agent", "target", maintain_order=True)
)
print(f"Number of (lab, video, agent, target) groups: {len(groups)}")

# ============================================================
# 2. Pre compute features for all videos
# ============================================================
print("Generating self and pair features for all test videos...")

rows = test_df.rows(named=True)

def process_video_features(row):
    lab_id = row["lab_id"]
    video_id = row["video_id"]

    tracking_path = TEST_TRACKING_DIR / f"{lab_id}/{video_id}.parquet"
    tracking = pl.read_parquet(tracking_path)

    self_feat = make_self_features(metadata=row, tracking=tracking)
    pair_feat = make_pair_features(metadata=row, tracking=tracking)

    self_feat.write_parquet(SELF_FEATURE_DIR / f"{video_id}.parquet")
    pair_feat.write_parquet(PAIR_FEATURE_DIR / f"{video_id}.parquet")

# Use Parallel to speed up feature generation
n_jobs = get_n_jobs()
print(f"Using {n_jobs} jobs for feature generation")
# Use threading backend to avoid pickling overhead and global variable issues
Parallel(n_jobs=n_jobs, backend="threading")(
    delayed(process_video_features)(row) for row in tqdm(rows, total=len(rows))
)

gc.collect()

# ============================================================
# 3. Inference by group and segment construction
# ============================================================
print("Running inference and building group submissions...")

def process_group_inference(group_data):
    (lab_id, video_id, agent, target), group_df = group_data
    return predict_for_group(
        lab_id=lab_id,
        video_id=video_id,
        agent=agent,
        target=target,
        group_behaviors=group_df,
    )

# Use Parallel to speed up inference
n_jobs = get_n_jobs()
print(f"Using {n_jobs} jobs for inference")

# Use threading backend to avoid pickling overhead and global variable issues
results = Parallel(n_jobs=n_jobs, backend="threading")(
    delayed(process_group_inference)((group_tuple, group_df))
    for group_tuple, group_df in tqdm(groups, total=len(groups))
)

group_submissions = [res for res in results if res is not None and res.height > 0]

if not group_submissions:
    raise RuntimeError(
        "No submissions were generated. "
        "Check that starter models exist under /kaggle/working/results."
    )

submission = pl.concat(group_submissions, how="vertical").sort(
    "video_id",
    "agent_id",
    "target_id",
    "action",
    "start_frame",
    "stop_frame",
)

print("Initial submission rows:", submission.height)

# ============================================================
# 4. Robustify and final clean up
# ============================================================
print("Running robustify on submission...")
submission = robustify(submission, test_df, train_test="test")

# Keep only valid intervals
submission = submission.filter(pl.col("start_frame") < pl.col("stop_frame"))

# Drop ultra short segments (likely noise)
submission = submission.with_columns(
    (pl.col("stop_frame") - pl.col("start_frame")).alias("duration")
).filter(pl.col("duration") >= 2).drop("duration")

print("Rows after robustify, validity check and duration filter:", submission.height)

# Add row_id and save as submission.csv
final_submission = submission.with_row_index("row_id")
final_path = WORKING_DIR / "submission.csv"
final_submission.write_csv(final_path)

print("Saved submission to:", final_path)
!head -n 10 /kaggle/working/submission.csv

Loading test metadata...
Building behavior table from behaviors_labeled...
Number of (lab, video, agent, target) groups: 16
Generating self and pair features for all test videos...
Using 2 jobs for feature generation


  0%|          | 0/1 [00:00<?, ?it/s]

Running inference and building group submissions...
Using 2 jobs for inference


  0%|          | 0/16 [00:00<?, ?it/s]

Initial submission rows: 1833
Running robustify on submission...
Rows after robustify, validity check and duration filter: 1176
Saved submission to: /kaggle/working/submission.csv
row_id,video_id,agent_id,target_id,action,start_frame,stop_frame
0,438887472,mouse4,mouse3,avoid,197,199
1,438887472,mouse4,mouse3,avoid,203,208
2,438887472,mouse4,mouse3,approach,214,230
3,438887472,mouse4,mouse3,chase,230,232
4,438887472,mouse4,mouse3,approach,232,235
5,438887472,mouse4,mouse3,attack,238,277
6,438887472,mouse4,mouse3,approach,529,532
7,438887472,mouse4,mouse3,avoid,577,579
8,438887472,mouse4,mouse3,avoid,721,743


In [2]:
# ============================================================
# 4.1 Merge nearby segments and drop tiny ones
#     (reduces fragmentation before summary) — Kaggle-safe save
# ============================================================
import os

MAX_GAP_FRAMES = int(os.getenv("MAX_GAP_FRAMES", 3))
MIN_DURATION_FRAMES = int(os.getenv("MIN_DURATION_FRAMES", 4))

print(
    f"Merging same-action segments with gaps <= {MAX_GAP_FRAMES} and dropping durations < {MIN_DURATION_FRAMES}"
)

pre_rows = submission.height

# Merge segments within each (video, agent, target, action)
keys = ["video_id", "agent_id", "target_id", "action"]

s = (
    submission
    .sort(keys + ["start_frame", "stop_frame"])  # stable order for windows
    .with_columns([
        pl.col("stop_frame").cast(pl.Int64),
        pl.col("start_frame").cast(pl.Int64),
    ])
    .with_columns([
        pl.col("stop_frame").shift(1).over(keys).alias("prev_stop"),
    ])
    .with_columns([
        (pl.col("start_frame") > (pl.col("prev_stop") + MAX_GAP_FRAMES))
        .fill_null(True)
        .cast(pl.Int32)
        .alias("is_new"),
    ])
    .with_columns([
        pl.col("is_new").cum_sum().over(keys).alias("block")
    ])
)

merged = (
    s
    .group_by(keys + ["block"], maintain_order=True)
    .agg([
        pl.col("start_frame").min().alias("start_frame"),
        pl.col("stop_frame").max().alias("stop_frame"),
    ])
    .select(keys + ["start_frame", "stop_frame"])  # drop helper cols
)

# Filter by minimum duration
submission = (
    merged
    .with_columns([
        pl.col("video_id").cast(pl.Int64),
        pl.col("start_frame").cast(pl.Int64),
        pl.col("stop_frame").cast(pl.Int64),
        (pl.col("stop_frame") - pl.col("start_frame")).alias("duration"),
    ])
    .filter(pl.col("duration") >= MIN_DURATION_FRAMES)
    .drop("duration")
    .sort(["video_id", "agent_id", "target_id", "action", "start_frame", "stop_frame"]) 
)

post_rows = submission.height
print(f"Segments before merge/filter: {pre_rows}; after: {post_rows}")

# Ensure exact column order and integer types for Kaggle
submission = submission.select([
    pl.col("video_id").cast(pl.Int64),
    pl.col("agent_id"),
    pl.col("target_id"),
    pl.col("action"),
    pl.col("start_frame").cast(pl.Int64),
    pl.col("stop_frame").cast(pl.Int64),
])

# Rebuild row_id 0..N-1 and save
final_submission = submission.with_row_index("row_id")
final_submission = final_submission.select([
    "row_id","video_id","agent_id","target_id","action","start_frame","stop_frame"
])
final_path = WORKING_DIR / "submission.csv"
final_submission.write_csv(final_path)
print("Re-saved merged submission to:", final_path)

Merging same-action segments with gaps <= 3 and dropping durations < 4
Segments before merge/filter: 1176; after: 699
Re-saved merged submission to: /kaggle/working/submission.csv


In [3]:
# ============================================================
# 5. Per‑video summary after merge
# ============================================================
print("Building per-video/action submission summary (post-merge)...")

from pathlib import Path
try:
    import polars as pl
except ImportError:
    raise ImportError("This cell requires polars.")

# Prefer in-memory 'submission'; fall back to saved CSV
try:
    sub_df = submission.clone()
except NameError:
    # Try Kaggle path first, else local
    kaggle_path = Path("/kaggle/working/submission.csv")
    local_path = Path("submission.csv")
    if kaggle_path.exists():
        sub_df = pl.read_csv(kaggle_path)
    elif local_path.exists():
        sub_df = pl.read_csv(local_path)
    else:
        raise FileNotFoundError("No submission.csv found at /kaggle/working or current directory.")
    if "row_id" in sub_df.columns:
        sub_df = sub_df.drop("row_id")

# Attach lab_id from test_df when available
if 'test_df' in globals():
    try:
        video_lab = test_df.select(["lab_id", "video_id"]).unique()
        sub_df = sub_df.join(video_lab, on="video_id", how="left")
    except Exception as e:
        print("Note: could not join lab_id:", e)
else:
    print("Note: test_df not in scope; skipping lab_id join")

# Ensure required columns exist
required_cols = {"video_id", "agent_id", "target_id", "action", "start_frame", "stop_frame"}
missing = required_cols - set(sub_df.columns)
if missing:
    raise ValueError(f"Submission is missing required columns: {missing}")

with_dur = sub_df.with_columns((pl.col("stop_frame") - pl.col("start_frame")).alias("duration"))

# Per video/action summary
by_cols = [c for c in ["lab_id", "video_id", "action"] if c in with_dur.columns]
vid_action_summary = (
    with_dur
    .group_by(by_cols)
    .agg([
        pl.len().alias("segments"),
        pl.col("duration").sum().alias("frames_total"),
        pl.col("duration").mean().alias("frames_avg"),
    ])
    .sort(by_cols + ["segments"], descending=[False]*len(by_cols) + [True])
)

print("Top 20 video/action rows by segment count:")
print(vid_action_summary.head(20))

# Overall action distribution
action_summary = (
    with_dur
    .group_by("action")
    .agg([
        pl.len().alias("segments"),
        pl.col("duration").sum().alias("frames_total"),
        pl.col("duration").mean().alias("frames_avg"),
    ])
    .sort(["segments"], descending=[True])
)

print("\nOverall action distribution (top 20):")
print(action_summary.head(20))


Building per-video/action submission summary (post-merge)...
Top 20 video/action rows by segment count:
shape: (7, 6)
┌────────────────┬───────────┬─────────────┬──────────┬──────────────┬────────────┐
│ lab_id         ┆ video_id  ┆ action      ┆ segments ┆ frames_total ┆ frames_avg │
│ ---            ┆ ---       ┆ ---         ┆ ---      ┆ ---          ┆ ---        │
│ str            ┆ i64       ┆ str         ┆ u32      ┆ i64          ┆ f64        │
╞════════════════╪═══════════╪═════════════╪══════════╪══════════════╪════════════╡
│ AdaptableSnail ┆ 438887472 ┆ approach    ┆ 80       ┆ 1164         ┆ 14.55      │
│ AdaptableSnail ┆ 438887472 ┆ attack      ┆ 39       ┆ 956          ┆ 24.512821  │
│ AdaptableSnail ┆ 438887472 ┆ avoid       ┆ 186      ┆ 3703         ┆ 19.908602  │
│ AdaptableSnail ┆ 438887472 ┆ chase       ┆ 37       ┆ 697          ┆ 18.837838  │
│ AdaptableSnail ┆ 438887472 ┆ chaseattack ┆ 1        ┆ 24           ┆ 24.0       │
│ AdaptableSnail ┆ 438887472 ┆ rear       